# Explore Mosqlimate Data

Test downloading and exploring the data made available through the Mosqlimate platform.

In [ ]:
import os
from pathlib import Path

import epiweeks
import mosqlient
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

print(os.getcwd())
ROOT_DIRPATH = Path("../..")

# ---- Helpers
def save_tmp_plotly_fig(_fig, _name, _dir=".local"):
    _fpath = ROOT_DIRPATH / Path(f"{_dir}/{_name}.html")
    _fpath.parent.mkdir(exist_ok=True, parents=True)
    _fig.write_html(_fpath, include_plotlyjs="cdn")


# Demographic data

REQUIRED to run the following cell groups

In [ ]:
imdc_regional_df = pd.read_csv("../../data/data_imdc_2026/map_regional_health.csv")
imdc_population_df = pd.read_csv("../../data/data_imdc_2026/datasus_population_2001_2025.csv.gz")


imdc_regional_df
imdc_population_df

## Dengue data

In [ ]:
df = pd.read_csv("../../data/data_imdc_2026/dengue.csv.gz", parse_dates=["date"])


imdc_dengue_df = df
df

In [ ]:
# Visualize state-level cases and case incidence
# -----------------------
df = imdc_dengue_df.copy()

# --- Add year and population data
df["year"] = df["date"].dt.year
df = df.merge(imdc_population_df, on=["geocode", "year"])


# --- Aggregate and calculate per-populaiton incidence
state_aggr_df = df.groupby(["uf", "date"]).agg({"casos": "sum", "population": "sum"})
# state_sr = df.groupby(["uf", "date"])["casos"].sum()

state_aggr_df["case_inc_100k"] = state_aggr_df["casos"] / state_aggr_df["population"] * 1E5

# ---- Plot abs cases
fig = px.line(state_aggr_df.reset_index(), x="date", y="casos", color="uf")
fig.show()
save_tmp_plotly_fig(fig, "dengue_cases_uf")

# --- Plot case incidence rate by 100k population
fig = px.line(state_aggr_df.reset_index(), x="date", y="case_inc_100k", color="uf")
fig.show()
save_tmp_plotly_fig(fig, "dengue_cases_inc_uf")


# Climate data

In [ ]:
df = pd.read_csv("../../data/data_imdc_2026/climate.csv.gz", parse_dates=["date"])

imdc_climate_df = df

imdc_climate_df


In [ ]:
df = imdc_climate_df

# --- Map municipalities into UFs
imdc_regional_df   # REQUIRES PREVIOUS CELL GROUP TO BE RUN
df = df.merge(imdc_regional_df[["geocode", "uf", "uf_name"]], on="geocode")

# # --- Add the year based on reference date
df["year"] = df["date"].dt.year

# --- Add population based on year and municipality
df = df.merge(imdc_population_df[:], on=["geocode", "year"])

# --- Aggreagate spatially by UF in multiple ways

named_aggs = dict()
#  ^ Dictionary of aggregations. Each entry is one aggregate with source columns, target variables and aggr function

# Simple average temperature, all municipalities treated equally
named_aggs["mean_temp_med"] = pd.NamedAgg(column="temp_med", aggfunc="mean")

# Weighted average temperature by municipality population
named_aggs["weighted_mean_temp_med"] = pd.NamedAgg(
    column="temp_med",
    aggfunc=lambda x: (x * df.loc[x.index, "population"]).sum() / df.loc[x.index, "population"].sum()
)


# -
spatial_agg_df = df.groupby(["date", "uf"]).agg(**named_aggs)
# spatial_agg_df = df.iloc[:10000].groupby(["date", "uf"]).agg(**named_aggs)  # ### PARTIAL, just to test

spatial_agg_df

In [ ]:
# Visualize variables
# ---------------
fig = px.line(
    spatial_agg_df.reset_index(),
    x="date", y="mean_temp_med", color="uf"
)



# --- Save
save_tmp_plotly_fig(fig, "mean_temp_med_uf")

fig.show()

# Compare different spatial temperature aggregation methods
# --------------
df = spatial_agg_df.copy()

df["temp_diff"] = df["weighted_mean_temp_med"] - df["mean_temp_med"]
df["temp_squared_error"] = df["temp_diff"]**2


df

# Box plot
fig = px.box(
    df.reset_index(), x="uf", y="temp_diff",
    title="Temperature Difference (Weighted minus Simple Average) by UF"
)
fig.show()
save_tmp_plotly_fig(fig, "temp_diff_uf")

# --- Dispersion plot for one UF
uf = "PR"
_df = df.xs(uf, level="uf")

fig = px.scatter(
    _df.reset_index(), x="mean_temp_med", y="weighted_mean_temp_med",
    title=f"Plain vs weighted average temperature for UF = {uf}"
)
fig.add_scatter(
    x=[15, 34], y=[15, 34], mode="lines", line_color="gray", line_dash="dash",
)

fig.show()
save_tmp_plotly_fig(fig, f"temp_dispersion_{uf}")

